In [32]:
import  pandas as pd
from pathlib import Path

In [33]:
from pathlib import Path
from itertools import combinations
from math import comb

import pandas as pd
from tqdm.auto import tqdm

In [40]:
from pathlib import Path
from itertools import combinations
from math import comb

import pandas as pd
from tqdm.auto import tqdm


def build_winner_loser_edges_from_collapsed(
    df_edges_collapsed,
    left_col="left_item",
    right_col="right_item",
    choice_col="more_vulnerable_household",
):
    """
    Build directed winner -> loser edges from collapsed pairwise rows.

    Household 1 = left_item wins
    Household 2 = right_item wins
    indeterminate = skipped
    """

    edges = []
    skipped_indeterminate = 0

    for _, row in df_edges_collapsed.iterrows():
        left_item = row[left_col]
        right_item = row[right_col]
        choice = str(row[choice_col]).strip().lower()

        if pd.isna(left_item) or pd.isna(right_item):
            continue

        if choice == "household 1":
            edges.append((left_item, right_item))

        elif choice == "household 2":
            edges.append((right_item, left_item))

        else:
            skipped_indeterminate += 1

    return edges, skipped_indeterminate


def naive_count_observable_triads_and_directed_cycles(
    edges,
    return_cycles=False,
    show_progress=True,
    desc="Counting triads",
):
    """
    Naively count observable undirected triangles and directed cyclic triads.

    For incomplete sampled graphs:

        T_max_sampled = number of 3-node sets where all 3 unordered edges exist

        T_n = number of those observable triads whose directions form a cycle

    Directed cycle patterns:
        a -> b -> c -> a
        a -> c -> b -> a

    Each 3-node triad is counted at most once in T_n.
    """

    nodes = sorted(
        set(u for u, v in edges) | set(v for u, v in edges),
        key=str,
    )

    adj = {node: set() for node in nodes}
    undirected_adj = {node: set() for node in nodes}

    for u, v in edges:
        if u == v:
            continue

        adj[u].add(v)

        undirected_adj[u].add(v)
        undirected_adj[v].add(u)

    observable_undirected_triads = 0
    directed_cyclic_triads = 0
    cycles = []

    total_triples = comb(len(nodes), 3) if len(nodes) >= 3 else 0

    triple_iter = combinations(nodes, 3)

    if show_progress:
        triple_iter = tqdm(
            triple_iter,
            total=total_triples,
            desc=desc,
            unit="triad",
        )

    for a, b, c in triple_iter:
        # Check whether all 3 unordered comparisons exist.
        is_observable_triangle = (
            b in undirected_adj[a]
            and c in undirected_adj[a]
            and c in undirected_adj[b]
        )

        if not is_observable_triangle:
            continue

        observable_undirected_triads += 1

        # Check the two possible directed 3-cycle orientations.
        cycle_orientation_1 = (
            b in adj[a]
            and c in adj[b]
            and a in adj[c]
        )

        cycle_orientation_2 = (
            c in adj[a]
            and b in adj[c]
            and a in adj[b]
        )

        if cycle_orientation_1 or cycle_orientation_2:
            directed_cyclic_triads += 1

            if return_cycles:
                if cycle_orientation_1:
                    cycles.append((a, b, c))
                else:
                    cycles.append((a, c, b))

    zeta = (
        1.0 - directed_cyclic_triads / observable_undirected_triads
        if observable_undirected_triads > 0
        else float("nan")
    )

    result = {
        "T_max_sampled_observable_undirected_triads": observable_undirected_triads,
        "T_n_directed_cyclic_triads": directed_cyclic_triads,
        "normalized_cycle_rate": (
            directed_cyclic_triads / observable_undirected_triads
            if observable_undirected_triads > 0
            else float("nan")
        ),
        "zeta": zeta,
    }

    if return_cycles:
        return result, cycles

    return result


def count_cycles_for_one_csv(csv_path, label=None, show_progress=True):
    """
    Load one parsed CSV, collapse repeated tie rows,
    build winner -> loser graph, count:

        T_max_sampled = observable undirected triangles
        T_n = directed cyclic triads
        zeta = 1 - T_n / T_max_sampled
    """

    csv_path = Path(csv_path)

    if label is None:
        label = csv_path.name

    print("\n==============================")
    print(f"Counting cycles for: {label}")
    print(f"Path: {csv_path}")
    print("==============================")

    df_edges = pd.read_csv(csv_path)
    df_edges_collapsed = collapse_repeated_tie_rows(df_edges)

    edges, skipped_indeterminate = build_winner_loser_edges_from_collapsed(
        df_edges_collapsed
    )

    num_edges_raw = len(edges)
    unique_edges = sorted(set(edges), key=lambda x: (str(x[0]), str(x[1])))
    num_edges_unique = len(unique_edges)

    nodes = sorted(
        set(u for u, v in unique_edges) | set(v for u, v in unique_edges),
        key=str,
    )

    observed_undirected_pairs = {
        frozenset((u, v))
        for u, v in unique_edges
        if u != v
    }

    num_observed_undirected_pairs = len(observed_undirected_pairs)

    # If this is > 0, it means both u -> v and v -> u appeared somewhere.
    num_bidirectional_pair_conflicts = (
        num_edges_unique - num_observed_undirected_pairs
    )

    triad_result = naive_count_observable_triads_and_directed_cycles(
        unique_edges,
        show_progress=show_progress,
        desc=f"Triads: {label}",
    )

    result = {
        "label": label,
        "csv_path": csv_path,
        "num_original_rows": len(df_edges),
        "num_collapsed_rows": len(df_edges_collapsed),
        "num_nodes": len(nodes),
        "num_possible_node_triads_nC3": comb(len(nodes), 3) if len(nodes) >= 3 else 0,
        "num_edges_raw": num_edges_raw,
        "num_edges_unique": num_edges_unique,
        "num_observed_undirected_pairs": num_observed_undirected_pairs,
        "num_duplicate_edges_removed": num_edges_raw - num_edges_unique,
        "num_bidirectional_pair_conflicts": num_bidirectional_pair_conflicts,
        "num_indeterminate_skipped": skipped_indeterminate,
        **triad_result,
    }

    print(f"Original rows:                         {result['num_original_rows']}")
    print(f"Collapsed rows:                        {result['num_collapsed_rows']}")
    print(f"Nodes:                                 {result['num_nodes']}")
    print(f"Possible node triads nC3:              {result['num_possible_node_triads_nC3']}")
    print(f"Raw winner->loser edges:               {result['num_edges_raw']}")
    print(f"Unique winner->loser edges:            {result['num_edges_unique']}")
    print(f"Observed undirected pairs:             {result['num_observed_undirected_pairs']}")
    print(f"Duplicate directed edges removed:      {result['num_duplicate_edges_removed']}")
    print(f"Bidirectional pair conflicts:          {result['num_bidirectional_pair_conflicts']}")
    print(f"Indeterminate skipped:                 {result['num_indeterminate_skipped']}")
    print(f"T_max sampled observable triads:       {result['T_max_sampled_observable_undirected_triads']}")
    print(f"T_n directed cyclic triads:            {result['T_n_directed_cyclic_triads']}")
    print(f"Normalized cycle rate T_n/T_max:       {result['normalized_cycle_rate']:.6f}")
    print(f"Zeta consistency coefficient:          {result['zeta']:.6f}")

    return result, df_edges_collapsed, unique_edges


def get_all_unique_csv_files(csvs, base_path):
    """
    From the nested csvs dict, collect every unique file path.
    Files are located at base_path / dataset / filename.
    """

    base_path = Path(base_path)
    all_files = []

    for dataset, file_pairs in csvs.items():
        for pair in file_pairs:
            for filename in pair:
                path = base_path / dataset / filename
                all_files.append((dataset, filename, path))

    # Remove duplicates while preserving order.
    seen = set()
    unique_files = []

    for dataset, filename, path in all_files:
        key = str(path)
        if key not in seen:
            seen.add(key)
            unique_files.append((dataset, filename, path))

    return unique_files


def count_cycles_for_all_csvs(csvs, base_path, show_progress=True):
    all_cycle_results = []
    all_collapsed_dfs = {}
    all_edges = {}

    unique_files = get_all_unique_csv_files(csvs, base_path)

    for dataset, filename, path in tqdm(
        unique_files,
        desc="Processing CSV files",
        unit="file",
    ):
        label = f"{dataset} / {filename}"

        result, df_edges_collapsed, unique_edges = count_cycles_for_one_csv(
            csv_path=path,
            label=label,
            show_progress=show_progress,
        )

        result["dataset"] = dataset
        result["filename"] = filename

        all_cycle_results.append(result)
        all_collapsed_dfs[(dataset, filename)] = df_edges_collapsed
        all_edges[(dataset, filename)] = unique_edges

    cycle_summary_df = pd.DataFrame(all_cycle_results)

    cycle_summary_df = cycle_summary_df[
        [
            "dataset",
            "filename",
            "num_nodes",
            "num_possible_node_triads_nC3",
            "num_original_rows",
            "num_collapsed_rows",
            "num_edges_raw",
            "num_edges_unique",
            "num_observed_undirected_pairs",
            "num_duplicate_edges_removed",
            "num_bidirectional_pair_conflicts",
            "num_indeterminate_skipped",
            "T_max_sampled_observable_undirected_triads",
            "T_n_directed_cyclic_triads",
            "normalized_cycle_rate",
            "zeta",
            "csv_path",
        ]
    ]

    return cycle_summary_df, all_collapsed_dfs, all_edges

In [ ]:
import pandas as pd


def collapse_repeated_tie_rows(df_edges: pd.DataFrame) -> pd.DataFrame:
	"""
	For each tie_index:
	  - If any repeat has a valid winner, keep the earliest valid repeat_index row.
	  - If all repeats are indeterminate, keep the earliest repeat_index row and mark it indeterminate.
	  - Adds winner_side and winner_item.
	"""

	required_cols = [
		"tie_index",
		"left_item",
		"right_item",
		"repeat_index",
		"more_vulnerable_household",
	]

	missing = [c for c in required_cols if c not in df_edges.columns]
	if missing:
		raise ValueError(f"Missing required columns: {missing}")

	df = df_edges.copy()

	# Make repeat_index sortable even if it was read as string
	df["repeat_index"] = pd.to_numeric(df["repeat_index"], errors="coerce")

	# Normalize winner text
	df["_choice_norm"] = (
		df["more_vulnerable_household"]
		.astype(str)
		.str.strip()
		.str.lower()
	)

	df["_is_valid_choice"] = df["_choice_norm"].isin(
		["household 1", "household 2"]
	)

	# Sort so earliest repeat_index appears first
	df = df.sort_values(
		["tie_index", "repeat_index"],
		kind="mergesort",
	)

	chosen_rows = []

	for tie_index, g in df.groupby("tie_index", sort=False):
		valid_g = g[g["_is_valid_choice"]]

		if len(valid_g) > 0:
			# Keep earliest valid repeat
			row = valid_g.iloc[0].copy()
			row["selection_status"] = "valid"
		else:
			# Keep one row and record as indeterminate
			row = g.iloc[0].copy()
			row["more_vulnerable_household"] = "indeterminate"
			row["_choice_norm"] = "indeterminate"
			row["selection_status"] = "indeterminate"

		choice = row["_choice_norm"]

		if choice == "household 1":
			row["winner_side"] = "left"
			row["winner_item"] = row["left_item"]
		elif choice == "household 2":
			row["winner_side"] = "right"
			row["winner_item"] = row["right_item"]
		else:
			row["winner_side"] = "indeterminate"
			row["winner_item"] = pd.NA

		chosen_rows.append(row)

	collapsed_df = pd.DataFrame(chosen_rows)

	# Drop helper columns
	collapsed_df = collapsed_df.drop(
		columns=["_choice_norm", "_is_valid_choice"],
		errors="ignore",
	)

	return collapsed_df.reset_index(drop=True)

In [ ]:
import numpy as np
import pandas as pd


def run_rank_centrality(
	df_edges_collapsed: pd.DataFrame,
	left_col="left_item",
	right_col="right_item",
	choice_col="more_vulnerable_household",
	indeterminate_policy="skip",   # "skip" or "tie"
	damping=0.01,                  # small damping helps if graph is not fully connected
	max_iter=10_000,
	tol=1e-12,
):
	"""
	Run Rank Centrality from collapsed pairwise comparison rows.

	Assumptions:
	  - Household 1 means left_item wins.
	  - Household 2 means right_item wins.
	  - indeterminate_policy:
		  "skip": ignore indeterminate comparisons
		  "tie": count as half-win for each side

	Returns:
	  ranking_df, transition_matrix, wins_matrix, item_list
	"""

	required_cols = [left_col, right_col, choice_col]
	missing = [c for c in required_cols if c not in df_edges_collapsed.columns]
	if missing:
		raise ValueError(f"Missing required columns: {missing}")

	df = df_edges_collapsed.copy()

	# Build item universe
	items = pd.Index(
		pd.concat([df[left_col], df[right_col]], ignore_index=True)
		.dropna()
		.unique()
	)

	item_to_idx = {item: idx for idx, item in enumerate(items)}
	n = len(items)

	print(f"Number of unique items: {n}")

	# wins[i, j] = number of times item i beat item j
	wins = np.zeros((n, n), dtype=float)

	skipped_indeterminate = 0

	for _, row in df.iterrows():
		left_item = row[left_col]
		right_item = row[right_col]

		if pd.isna(left_item) or pd.isna(right_item):
			continue

		i = item_to_idx[left_item]
		j = item_to_idx[right_item]

		choice = str(row[choice_col]).strip().lower()

		if choice == "household 1":
			# left beats right
			wins[i, j] += 1.0

		elif choice == "household 2":
			# right beats left
			wins[j, i] += 1.0

		else:
			# indeterminate
			if indeterminate_policy == "tie":
				wins[i, j] += 0.5
				wins[j, i] += 0.5
			elif indeterminate_policy == "skip":
				skipped_indeterminate += 1
			else:
				raise ValueError("indeterminate_policy must be 'skip' or 'tie'")

	print(f"Skipped indeterminate comparisons: {skipped_indeterminate}")

	# comparisons[i, j] = total comparisons observed between i and j
	comparisons = wins + wins.T

	# Degree = how many unique other items each item was compared with
	degrees = (comparisons > 0).sum(axis=1)
	d_max = max(degrees.max(), 1)

	print(f"Max comparison degree: {d_max}")

	# Build Rank Centrality transition matrix
	# P[i, j] = probability of moving from i to j
	# We move toward j if j beats i.
	P = np.zeros((n, n), dtype=float)

	for i in range(n):
		for j in range(n):
			if i == j:
				continue

			total_ij = comparisons[i, j]

			if total_ij > 0:
				prob_j_beats_i = wins[j, i] / total_ij
				P[i, j] = prob_j_beats_i / d_max

		P[i, i] = 1.0 - P[i].sum()

	# Optional damping for numerical stability / disconnected graphs
	if damping is not None and damping > 0:
		P = (1.0 - damping) * P + damping * np.ones((n, n)) / n

	# Power iteration to get stationary distribution
	pi = np.ones(n) / n

	for iteration in range(max_iter):
		new_pi = pi @ P

		if np.linalg.norm(new_pi - pi, ord=1) < tol:
			pi = new_pi
			print(f"Converged after {iteration + 1} iterations")
			break

		pi = new_pi
	else:
		print("Warning: power iteration did not fully converge")

	num_wins = wins.sum(axis=1)
	num_losses = wins.sum(axis=0)
	num_comparisons = num_wins + num_losses

	ranking_df = pd.DataFrame(
		{
			"item": items,
			"rank_centrality_score": pi,
			"num_wins": num_wins,
			"num_losses": num_losses,
			"num_comparisons": num_comparisons,
		}
	)

	ranking_df["win_rate"] = np.where(
		ranking_df["num_comparisons"] > 0,
		ranking_df["num_wins"] / ranking_df["num_comparisons"],
		np.nan,
	)

	ranking_df = ranking_df.sort_values(
		"rank_centrality_score",
		ascending=False,
	).reset_index(drop=True)

	ranking_df["rank"] = np.arange(1, len(ranking_df) + 1)

	return ranking_df, P, wins, items

In [ ]:
import pandas as pd
from scipy.stats import spearmanr


In [24]:
from pathlib import Path
import pandas as pd
from scipy.stats import spearmanr


def compare_two_rank_centrality_csvs(
	csv_paths,
	labels=None,
	indeterminate_policy="skip",
	damping=0.01,
):
	"""
	Takes a list of 2 parsed CSV paths.
	
	For each file:
	  - loads df_edges
	  - collapses repeated tie_index rows
	  - reports shape and indeterminate count
	  - constructs Rank Centrality ranking
	
	Then:
	  - reports missing tie_index rows between files
	  - computes Spearman rho between the two rankings
	
	Returns a dictionary containing all intermediate outputs and correlations.
	"""

	if len(csv_paths) != 2:
		raise ValueError("csv_paths must be a list of exactly 2 paths.")

	csv_paths = [Path(p) for p in csv_paths]

	if labels is None:
		labels = ["File 1", "File 2"]

	def build_ranking_from_csv(csv_path, label):
		print("\n==============================")
		print(f"Loading {label}")
		print(f"Path: {csv_path}")
		print("==============================")

		df_edges = pd.read_csv(csv_path)

		df_edges_collapsed = collapse_repeated_tie_rows(df_edges)

		num_indeterminate = (
			df_edges_collapsed["more_vulnerable_household"]
			.astype(str)
			.str.strip()
			.str.lower()
			.eq("indeterminate")
			.sum()
		)

		print(f"{label} original shape:     {df_edges.shape}")
		print(f"{label} collapsed shape:    {df_edges_collapsed.shape}")
		print(f"{label} unique tie_index:   {df_edges_collapsed['tie_index'].nunique()}")
		print(f"{label} indeterminate:      {num_indeterminate}")

		ranking_df, P, wins, items = run_rank_centrality(
			df_edges_collapsed,
			indeterminate_policy=indeterminate_policy,
			damping=damping,
		)

		return {
			"df_edges": df_edges,
			"df_edges_collapsed": df_edges_collapsed,
			"ranking_df": ranking_df,
			"P": P,
			"wins": wins,
			"items": items,
			"num_indeterminate": num_indeterminate,
		}

	# -------------------------
	# Build both rankings
	# -------------------------
	result_1 = build_ranking_from_csv(csv_paths[0], labels[0])
	result_2 = build_ranking_from_csv(csv_paths[1], labels[1])

	df_edges_collapsed_1 = result_1["df_edges_collapsed"]
	df_edges_collapsed_2 = result_2["df_edges_collapsed"]

	ranking_df_1 = result_1["ranking_df"]
	ranking_df_2 = result_2["ranking_df"]

	# -------------------------
	# Missing tie_index rows
	# -------------------------
	tie_ids_1 = set(df_edges_collapsed_1["tie_index"])
	tie_ids_2 = set(df_edges_collapsed_2["tie_index"])

	missing_from_file_1 = sorted(tie_ids_2 - tie_ids_1)
	missing_from_file_2 = sorted(tie_ids_1 - tie_ids_2)

	print("\n==============================")
	print("Missing tie_index comparison")
	print("==============================")
	print(f"In {labels[1]} but missing from {labels[0]}: {len(missing_from_file_1)}")
	print(f"In {labels[0]} but missing from {labels[1]}: {len(missing_from_file_2)}")

	print(f"\nFirst few missing from {labels[0]}:")
	print(missing_from_file_1[:20])

	print(f"\nFirst few missing from {labels[1]}:")
	print(missing_from_file_2[:20])

	# -------------------------
	# Spearman rho
	# -------------------------
	r1 = ranking_df_1[["item", "rank", "rank_centrality_score"]].copy()
	r2 = ranking_df_2[["item", "rank", "rank_centrality_score"]].copy()

	r1 = r1.rename(
		columns={
			"rank": "rank_1",
			"rank_centrality_score": "score_1",
		}
	)

	r2 = r2.rename(
		columns={
			"rank": "rank_2",
			"rank_centrality_score": "score_2",
		}
	)

	merged_rankings = r1.merge(r2, on="item", how="inner")

	rho_rank, pval_rank = spearmanr(
		merged_rankings["rank_1"],
		merged_rankings["rank_2"],
	)

	rho_score, pval_score = spearmanr(
		merged_rankings["score_1"],
		merged_rankings["score_2"],
	)
	
	tau_rank, pval_rank = kendalltau(
	merged_rankings["rank_1"],
	merged_rankings["rank_2"],
	)

	tau_score, pval_score = kendalltau(
		merged_rankings["score_1"],
		merged_rankings["score_2"],
	)



	print("\n==============================")
	print("Spearman correlation")
	print("==============================")
	print(f"Common ranked items:          {len(merged_rankings)}")
	print(f"Spearman rho using ranks:     {rho_rank:.6f}")
	print(f"p-value using ranks:          {pval_rank:.6g}")
	print(f"Spearman rho using scores:    {rho_score:.6f}")
	print(f"p-value using scores:         {pval_score:.6g}")

	print("\n==============================")
	print("Kendall tau correlation")
	print("==============================")
	print(f"Common ranked items:          {len(merged_rankings)}")
	print(f"Kendall tau using ranks:      {tau_rank:.6f}")
	print(f"p-value using ranks:          {pval_rank:.6g}")
	print(f"Kendall tau using scores:     {tau_score:.6f}")
	print(f"p-value using scores:         {pval_score:.6g}")

	return {
		"csv_paths": csv_paths,
		"labels": labels,

		"df_edges_list": [
			result_1["df_edges"],
			result_2["df_edges"],
		],

		"df_edges_collapsed_list": [
			result_1["df_edges_collapsed"],
			result_2["df_edges_collapsed"],
		],

		"ranking_dfs": [
			result_1["ranking_df"],
			result_2["ranking_df"],
		],

		"P_list": [
			result_1["P"],
			result_2["P"],
		],

		"wins_list": [
			result_1["wins"],
			result_2["wins"],
		],

		"items_list": [
			result_1["items"],
			result_2["items"],
		],

		"missing": {
			"missing_from_file_1": missing_from_file_1,
			"missing_from_file_2": missing_from_file_2,
			"num_missing_from_file_1": len(missing_from_file_1),
			"num_missing_from_file_2": len(missing_from_file_2),
		},

		"correlations": {
			"rho_rank": rho_rank,
			"pval_rank": pval_rank,
			"rho_score": rho_score,
			"pval_score": pval_score,
			"num_common_items": len(merged_rankings),
		},

		"merged_rankings": merged_rankings,
	}

In [ ]:
from pathlib import Path
import pandas as pd

In [28]:
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr, kendalltau


base_path = Path("parsed_outputs")

csvs = {
    "VISPDAT": [
        ["AIES_QWEN_vispdat_NewFixedLogger_parsed.csv", "AIES_vispdat_QWEN_NewFixedLoggerRun1SeedFixed_parsed.csv"],
        ["AIES_vispdat_llama7_NewFixedLoggerRun1SeedFixed_parsed.csv", "AIES_vispdat_llama7_NewFixedLogger_parsed.csv"],
        ["AIES_vispdat_DS7_NewFixedLogger_parsed.csv", "AIES_vispdat_DS8_NewFixedLoggerRun1SeedFixed_parsed.csv"],
    ],
    "VIFSPDAT": [
        ["AIES_QWEN_vifspdat_NewFixedLogger_parsed.csv", "AIES_vifspdat_qwen_NewFixedLoggerRun1SeedFixed_parsed.csv"],
        ["AIES_vifspdat_llama7_NewFixedLogger_parsed.csv", "AIES_vifspdat_llama7_NewFixedLoggerRun1SeedFixed_parsed.csv"],
        ["AIES_vifspdat_deepseek8B_NewFixedLoggerRun1SeedFixed_parsed.csv", "AIES_vifspdat_DS7_NewFixedLogger_parsed.csv"],
    ],
    "TAYVISPDAT": [
        ["AIES_QWEN_TAYvispdat_NewFixedLogger_parsed.csv", "AIES_tayvispdat_qwen_NewFixedLoggerRun1SeedFixed_parsed.csv"],
        ["AIES_TAYvispdat_llama7_NewFixedLogger_parsed.csv", "AIES_tayvispdat_llama7_NewFixedLoggerRun1SeedFixed_parsed.csv"],
        ["AIES_tayvispdat_DS7_NewFixedLogger_parsed.csv", "AIES_tayvispdat_DS8_NewFixedLoggerRun1SeedFixed_parsed.csv"],
    ],
}


def infer_model_from_filenames(file_pair):
    joined = " ".join(file_pair).lower()

    if "qwen" in joined:
        return "Qwen"
    if "llama" in joined:
        return "LLaMA"
    if "deepseek" in joined or "ds7" in joined or "ds8" in joined:
        return "DeepSeek"

    return "Unknown"


def run_all_rank_centrality_comparisons(
    csvs,
    base_path,
    indeterminate_policy="skip",
    damping=0.01,
):
    all_results = {}
    summary_rows = []

    for dataset, file_pairs in csvs.items():
        all_results[dataset] = {}

        for file_pair in file_pairs:
            model = infer_model_from_filenames(file_pair)

            csv_paths = [
                base_path / dataset / file_pair[0],
                base_path / dataset / file_pair[1],
            ]

            labels = [
                f"{dataset} {model} File 1",
                f"{dataset} {model} File 2",
            ]

            print("\n\n##################################################")
            print(f"Dataset: {dataset}")
            print(f"Model:   {model}")
            print("##################################################")

            results = compare_two_rank_centrality_csvs(
                csv_paths=csv_paths,
                labels=labels,
                indeterminate_policy=indeterminate_policy,
                damping=damping,
            )

            merged_rankings = results["merged_rankings"]

            # Spearman rho on aligned ranks
            spearman_rank, spearman_rank_p = spearmanr(
                merged_rankings["rank_1"],
                merged_rankings["rank_2"],
            )

            # Kendall tau on aligned ranks
            kendall_rank, kendall_rank_p = kendalltau(
                merged_rankings["rank_1"],
                merged_rankings["rank_2"],
            )

            # Optional: correlations on Rank Centrality scores
            spearman_score, spearman_score_p = spearmanr(
                merged_rankings["score_1"],
                merged_rankings["score_2"],
            )

            kendall_score, kendall_score_p = kendalltau(
                merged_rankings["score_1"],
                merged_rankings["score_2"],
            )

            summary_rows.append(
                {
                    "dataset": dataset,
                    "model": model,
                    "file_1": file_pair[0],
                    "file_2": file_pair[1],
                    "num_common_items": len(merged_rankings),
                    "missing_from_file_1": results["missing"]["num_missing_from_file_1"],
                    "missing_from_file_2": results["missing"]["num_missing_from_file_2"],
                    "spearman_rho_rank": spearman_rank,
                    "spearman_rank_p": spearman_rank_p,
                    "kendall_tau_rank": kendall_rank,
                    "kendall_rank_p": kendall_rank_p,
                    "spearman_rho_score": spearman_score,
                    "spearman_score_p": spearman_score_p,
                    "kendall_tau_score": kendall_score,
                    "kendall_score_p": kendall_score_p,
                }
            )

            all_results[dataset][model] = results

    summary_df = pd.DataFrame(summary_rows)

    return summary_df, all_results

In [29]:
summary_df, all_results = run_all_rank_centrality_comparisons(
    csvs=csvs,
    base_path=base_path,
    indeterminate_policy="skip",
    damping=0.01,
)

summary_df



##################################################
Dataset: VISPDAT
Model:   Qwen
##################################################

Loading VISPDAT Qwen File 1
Path: parsed_outputs\VISPDAT\AIES_QWEN_vispdat_NewFixedLogger_parsed.csv
VISPDAT Qwen File 1 original shape:     (42120, 23)
VISPDAT Qwen File 1 collapsed shape:    (21060, 26)
VISPDAT Qwen File 1 unique tie_index:   21060
VISPDAT Qwen File 1 indeterminate:      18
Number of unique items: 325
Skipped indeterminate comparisons: 18
Max comparison degree: 156
Converged after 1641 iterations

Loading VISPDAT Qwen File 2
Path: parsed_outputs\VISPDAT\AIES_vispdat_QWEN_NewFixedLoggerRun1SeedFixed_parsed.csv
VISPDAT Qwen File 2 original shape:     (42120, 23)
VISPDAT Qwen File 2 collapsed shape:    (21060, 26)
VISPDAT Qwen File 2 unique tie_index:   21060
VISPDAT Qwen File 2 indeterminate:      22
Number of unique items: 325
Skipped indeterminate comparisons: 22
Max comparison degree: 156
Converged after 1679 iterations

Missing tie

,dataset,model,file_1,file_2,num_common_items,missing_from_file_1,missing_from_file_2,spearman_rho_rank,spearman_rank_p,kendall_tau_rank,kendall_rank_p,spearman_rho_score,spearman_score_p,kendall_tau_score,kendall_score_p
0,VISPDAT,Qwen,AIES_QWEN_vispdat_NewFixedLogger_parsed.csv,AIES_vispdat_QWEN_NewFixedLoggerRun1SeedFixed_...,325,0,0,0.985454,1.094156e-250,0.902564,3.511439e-130,0.985454,1.094156e-250,0.902564,3.511439e-130
1,VISPDAT,LLaMA,AIES_vispdat_llama7_NewFixedLoggerRun1SeedFixe...,AIES_vispdat_llama7_NewFixedLogger_parsed.csv,325,0,0,0.978953,5.330489e-225,0.897892,7.400306e-129,0.978953,5.330489e-225,0.897892,7.400306e-129
2,VISPDAT,DeepSeek,AIES_vispdat_DS7_NewFixedLogger_parsed.csv,AIES_vispdat_DS8_NewFixedLoggerRun1SeedFixed_p...,325,0,0,0.966797,1.878278e-193,0.852877,1.870049e-116,0.966797,1.878278e-193,0.852877,1.870049e-116
3,VIFSPDAT,Qwen,AIES_QWEN_vifspdat_NewFixedLogger_parsed.csv,AIES_vifspdat_qwen_NewFixedLoggerRun1SeedFixed...,698,0,0,0.993570,0.000000e+00,0.930426,3.836760e-296,0.993570,0.000000e+00,0.930426,3.836760e-296
4,VIFSPDAT,LLaMA,AIES_vifspdat_llama7_NewFixedLogger_parsed.csv,AIES_vifspdat_llama7_NewFixedLoggerRun1SeedFix...,698,0,3221,0.973993,0.000000e+00,0.879903,4.409040e-265,0.973993,0.000000e+00,0.879903,4.409040e-265
5,VIFSPDAT,DeepSeek,AIES_vifspdat_deepseek8B_NewFixedLoggerRun1See...,AIES_vifspdat_DS7_NewFixedLogger_parsed.csv,698,0,0,0.978069,0.000000e+00,0.884729,5.650475e-268,0.978069,0.000000e+00,0.884729,5.650475e-268
6,TAYVISPDAT,Qwen,AIES_QWEN_TAYvispdat_NewFixedLogger_parsed.csv,AIES_tayvispdat_qwen_NewFixedLoggerRun1SeedFix...,561,0,0,0.992372,0.000000e+00,0.924599,3.336693e-235,0.992372,0.000000e+00,0.924599,3.336693e-235
7,TAYVISPDAT,LLaMA,AIES_TAYvispdat_llama7_NewFixedLogger_parsed.csv,AIES_tayvispdat_llama7_NewFixedLoggerRun1SeedF...,561,0,0,0.990067,0.000000e+00,0.923300,1.505324e-234,0.990067,0.000000e+00,0.923300,1.505324e-234
8,TAYVISPDAT,DeepSeek,AIES_tayvispdat_DS7_NewFixedLogger_parsed.csv,AIES_tayvispdat_DS8_NewFixedLoggerRun1SeedFixe...,561,0,0,0.977759,0.000000e+00,0.887179,1.020214e-216,0.977759,0.000000e+00,0.887179,1.020214e-216


In [48]:
cols_to_print = [
    "dataset",
    "model",
    "spearman_rho_rank",
    "spearman_rank_p",
    "kendall_tau_rank",
    "kendall_rank_p"
]

summary_df[cols_to_print]

,dataset,model,spearman_rho_rank,spearman_rank_p,kendall_tau_rank,kendall_rank_p
0,VISPDAT,Qwen,0.985454,1.094156e-250,0.902564,3.511439e-130
1,VISPDAT,LLaMA,0.978953,5.330489e-225,0.897892,7.400306e-129
2,VISPDAT,DeepSeek,0.966797,1.878278e-193,0.852877,1.870049e-116
3,VIFSPDAT,Qwen,0.993570,0.000000e+00,0.930426,3.836760e-296
4,VIFSPDAT,LLaMA,0.973993,0.000000e+00,0.879903,4.409040e-265
5,VIFSPDAT,DeepSeek,0.978069,0.000000e+00,0.884729,5.650475e-268
6,TAYVISPDAT,Qwen,0.992372,0.000000e+00,0.924599,3.336693e-235
7,TAYVISPDAT,LLaMA,0.990067,0.000000e+00,0.923300,1.505324e-234
8,TAYVISPDAT,DeepSeek,0.977759,0.000000e+00,0.887179,1.020214e-216


cycle counts

In [41]:
cycle_summary_df, all_collapsed_dfs, all_edges = count_cycles_for_all_csvs(
    csvs=csvs,
    base_path=base_path,
    show_progress=True,
)

cycle_summary_df

Processing CSV files:   0%|          | 0/18 [00:00<?, ?file/s]


Counting cycles for: VISPDAT / AIES_QWEN_vispdat_NewFixedLogger_parsed.csv
Path: parsed_outputs\VISPDAT\AIES_QWEN_vispdat_NewFixedLogger_parsed.csv


Triads: VISPDAT / AIES_QWEN_vispdat_NewFixedLogger_parsed.csv:   0%|          | 0/5668650 [00:00<?, ?triad/s]

Original rows:                         42120
Collapsed rows:                        21060
Nodes:                                 325
Possible node triads nC3:              5668650
Raw winner->loser edges:               21042
Unique winner->loser edges:            21042
Observed undirected pairs:             21042
Duplicate directed edges removed:      0
Bidirectional pair conflicts:          0
Indeterminate skipped:                 18
T_max sampled observable triads:       362090
T_n directed cyclic triads:            10845
Normalized cycle rate T_n/T_max:       0.029951
Zeta consistency coefficient:          0.970049

Counting cycles for: VISPDAT / AIES_vispdat_QWEN_NewFixedLoggerRun1SeedFixed_parsed.csv
Path: parsed_outputs\VISPDAT\AIES_vispdat_QWEN_NewFixedLoggerRun1SeedFixed_parsed.csv


Triads: VISPDAT / AIES_vispdat_QWEN_NewFixedLoggerRun1SeedFixed_parsed.csv:   0%|          | 0/5668650 [00:00<…

...

Triads: TAYVISPDAT / AIES_tayvispdat_DS7_NewFixedLogger_parsed.csv:   0%|          | 0/29269240 [00:00<?, ?tri…

Original rows:                         125664
Collapsed rows:                        62832
Nodes:                                 561
Possible node triads nC3:              29269240
Raw winner->loser edges:               62583
Unique winner->loser edges:            62583
Observed undirected pairs:             62583
Duplicate directed edges removed:      0
Bidirectional pair conflicts:          0
Indeterminate skipped:                 249
T_max sampled observable triads:       1850307
T_n directed cyclic triads:            70157
Normalized cycle rate T_n/T_max:       0.037916
Zeta consistency coefficient:          0.962084

Counting cycles for: TAYVISPDAT / AIES_tayvispdat_DS8_NewFixedLoggerRun1SeedFixed_parsed.csv
Path: parsed_outputs\TAYVISPDAT\AIES_tayvispdat_DS8_NewFixedLoggerRun1SeedFixed_parsed.csv


Triads: TAYVISPDAT / AIES_tayvispdat_DS8_NewFixedLoggerRun1SeedFixed_parsed.csv:   0%|          | 0/29269240 […

Original rows:                         125664
Collapsed rows:                        62832
Nodes:                                 561
Possible node triads nC3:              29269240
Raw winner->loser edges:               62574
Unique winner->loser edges:            62574
Observed undirected pairs:             62574
Duplicate directed edges removed:      0
Bidirectional pair conflicts:          0
Indeterminate skipped:                 258
T_max sampled observable triads:       1851242
T_n directed cyclic triads:            68483
Normalized cycle rate T_n/T_max:       0.036993
Zeta consistency coefficient:          0.963007


,dataset,filename,num_nodes,num_possible_node_triads_nC3,num_original_rows,num_collapsed_rows,num_edges_raw,num_edges_unique,num_observed_undirected_pairs,num_duplicate_edges_removed,num_bidirectional_pair_conflicts,num_indeterminate_skipped,T_max_sampled_observable_undirected_triads,T_n_directed_cyclic_triads,normalized_cycle_rate,zeta,csv_path
0,VISPDAT,AIES_QWEN_vispdat_NewFixedLogger_parsed.csv,325,5668650,42120,21060,21042,21042,21042,0,0,18,362090,10845,0.029951,0.970049,parsed_outputs\VISPDAT\AIES_QWEN_vispdat_NewFi...
1,VISPDAT,AIES_vispdat_QWEN_NewFixedLoggerRun1SeedFixed_...,325,5668650,42120,21060,21038,21038,21038,0,0,22,361195,11568,0.032027,0.967973,parsed_outputs\VISPDAT\AIES_vispdat_QWEN_NewFi...
2,VISPDAT,AIES_vispdat_llama7_NewFixedLoggerRun1SeedFixe...,325,5668650,42120,21060,21060,21060,21060,0,0,0,362340,6530,0.018022,0.981978,parsed_outputs\VISPDAT\AIES_vispdat_llama7_New...
3,VISPDAT,AIES_vispdat_llama7_NewFixedLogger_parsed.csv,325,5668650,42120,21060,21059,21059,21059,0,0,1,362927,6516,0.017954,0.982046,parsed_outputs\VISPDAT\AIES_vispdat_llama7_New...
4,VISPDAT,AIES_vispdat_DS7_NewFixedLogger_parsed.csv,325,5668650,42120,21060,19866,19866,19866,0,0,1194,305681,12740,0.041677,0.958323,parsed_outputs\VISPDAT\AIES_vispdat_DS7_NewFix...
5,VISPDAT,AIES_vispdat_DS8_NewFixedLoggerRun1SeedFixed_p...,325,5668650,42120,21060,19770,19770,19770,0,0,1290,300808,11968,0.039786,0.960214,parsed_outputs\VISPDAT\AIES_vispdat_DS8_NewFix...
6,VIFSPDAT,AIES_QWEN_vifspdat_NewFixedLogger_parsed.csv,698,56434696,194602,97301,97294,97294,97294,0,0,7,3608693,188175,0.052145,0.947855,parsed_outputs\VIFSPDAT\AIES_QWEN_vifspdat_New...
7,VIFSPDAT,AIES_vifspdat_qwen_NewFixedLoggerRun1SeedFixed...,698,56434696,194602,97301,97292,97292,97292,0,0,9,3614874,186441,0.051576,0.948424,parsed_outputs\VIFSPDAT\AIES_vifspdat_qwen_New...
8,VIFSPDAT,AIES_vifspdat_llama7_NewFixedLogger_parsed.csv,698,56434696,194602,97301,97276,97276,97276,0,0,25,3606627,116376,0.032267,0.967733,parsed_outputs\VIFSPDAT\AIES_vifspdat_llama7_N...
9,VIFSPDAT,AIES_vifspdat_llama7_NewFixedLoggerRun1SeedFix...,698,56434696,188160,94080,94065,94065,94065,0,0,15,3374872,117241,0.034739,0.965261,parsed_outputs\VIFSPDAT\AIES_vifspdat_llama7_N...


In [46]:
row_labels = ["QWEN 1", "QWEN 2", "LLaMA 1", "LLaMA 2", "DS1", "DS2"]

cycle_summary_df["model_run"] = (
    cycle_summary_df
    .groupby("dataset")
    .cumcount()
    .map(lambda i: row_labels[i])
)

cols_to_print = [
    "dataset",
    "model_run",
    "T_max_sampled_observable_undirected_triads",
    "T_n_directed_cyclic_triads",
    "zeta",
]

print(
    cycle_summary_df[cols_to_print]
    .round(
        {
            "normalized_cycle_rate": 6,
            "zeta": 6,
        }
    )
    .to_string(index=False)
)

   dataset model_run  T_max_sampled_observable_undirected_triads  T_n_directed_cyclic_triads     zeta
   VISPDAT    QWEN 1                                      362090                       10845 0.970049
   VISPDAT    QWEN 2                                      361195                       11568 0.967973
   VISPDAT   LLaMA 1                                      362340                        6530 0.981978
   VISPDAT   LLaMA 2                                      362927                        6516 0.982046
   VISPDAT       DS1                                      305681                       12740 0.958323
   VISPDAT       DS2                                      300808                       11968 0.960214
  VIFSPDAT    QWEN 1                                     3608693                      188175 0.947855
  VIFSPDAT    QWEN 2                                     3614874                      186441 0.948424
  VIFSPDAT   LLaMA 1                                     3606627                  

In [47]:
cycle_summary_df[cols_to_print].round(
        {
            "normalized_cycle_rate": 6,
            "zeta": 6,
        }
    )

,dataset,model_run,T_max_sampled_observable_undirected_triads,T_n_directed_cyclic_triads,zeta
0,VISPDAT,QWEN 1,362090,10845,0.970049
1,VISPDAT,QWEN 2,361195,11568,0.967973
2,VISPDAT,LLaMA 1,362340,6530,0.981978
3,VISPDAT,LLaMA 2,362927,6516,0.982046
4,VISPDAT,DS1,305681,12740,0.958323
5,VISPDAT,DS2,300808,11968,0.960214
6,VIFSPDAT,QWEN 1,3608693,188175,0.947855
7,VIFSPDAT,QWEN 2,3614874,186441,0.948424
8,VIFSPDAT,LLaMA 1,3606627,116376,0.967733
9,VIFSPDAT,LLaMA 2,3374872,117241,0.965261
